# Sistem Case-Based Reasoning (CBR) untuk Analisis Putusan Pengadilan TPPO

**Mata Kuliah:** Penalaran Komputer B
**Domain:** Pidana Khusus - Perdagangan Orang (TPPO)
**Tim:** Moh. Khairul Umam (202310370311448) & Nisrina Nurhafizhah (202310370311321)

**Tahapan CBR:**
1. **Preprocessing** - Ekstraksi dan pembersihan teks dari PDF
2. **Case Representation** - Ekstraksi metadata dan fitur
3. **Case Retrieval** - TF-IDF + Cosine Similarity & SVM/Naive Bayes (train/test split)
4. **Solution Reuse** - Prediksi hasil putusan berbasis weighted similarity voting
5. **Evaluasi** - Metrik retrieval, prediksi, dan visualisasi perbandingan

---
## Tahap 1: Preprocessing - Ekstraksi dan Pembersihan Teks PDF

In [ ]:
import os
import re
import glob
import json
import pickle
import pdfplumber
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

try:
    BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
except:
    BASE_DIR = os.getcwd()

# Path relatif: notebook di sub-cpmk-2-CBR/notebooks/, PDF di Tugas 3/pdf/
PROJECT_ROOT = os.path.dirname(BASE_DIR)  # sub-cpmk-2-CBR/
PDF_DIR = os.path.join(PROJECT_ROOT, '..', 'pdf')  # Tugas 3/pdf/
RAW_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
EVAL_DIR = os.path.join(PROJECT_ROOT, 'data', 'eval')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'data', 'results')
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Library berhasil dimuat.')

In [ ]:
def extract_clean_text_from_pdf(pdf_path):
    """Ekstrak teks dari PDF dengan filter font untuk menghapus watermark."""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            all_lines = []
            for page in pdf.pages:
                chars = page.chars
                if not chars:
                    continue
                real_chars = [
                    c for c in chars
                    if 6.0 <= round(c.get('size', 0), 1) <= 40.0
                ]
                if not real_chars:
                    continue
                y_lines = {}
                for c in real_chars:
                    y_key = round(c['y0'], 0)
                    if y_key not in y_lines:
                        y_lines[y_key] = []
                    y_lines[y_key].append(c)
                for y_key in sorted(y_lines, reverse=True):
                    line_chars = sorted(y_lines[y_key], key=lambda c: c['x0'])
                    line_text = ''.join(c['text'] for c in line_chars)
                    all_lines.append(line_text)
            return '\n'.join(all_lines) if all_lines else ''
    except Exception as e:
        print(f"  Error ekstraksi PDF: {e}")
        return None


def post_clean_text(text):
    """Pembersihan lanjutan: header, footer, email, scattered chars."""
    lines = text.split('\n')
    cleaned = []
    skip_until_empty = False
    for line in lines:
        stripped = line.strip()
        if not stripped:
            skip_until_empty = False
            continue
        if 'Disclaimer' in stripped:
            skip_until_empty = True
            continue
        if skip_until_empty:
            continue
        if stripped == 'P U T U S A N':
            continue
        if stripped.startswith('M A H K A M A H'):
            continue
        if 'Direktori Putusan Mahkamah Agung' in stripped:
            continue
        if 'putusan.mahkamahagung.go.id' in stripped:
            continue
        if stripped == 'DEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA':
            continue
        if re.match(r'^Halaman \d+ dari \d+ halaman', stripped):
            continue
        if re.match(r'^Halaman \d+$', stripped):
            continue
        if 'Kepaniteraan Mahkamah Agung' in stripped:
            skip_until_empty = True
            continue
        if re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.(go\.id|com|org)$', stripped):
            continue
        if re.match(r'^\d{2,3}-\d{3,4}-\d{4}$', stripped):
            continue
        words = stripped.split()
        meaningful = [w for w in words if len(w) > 1 or w.isdigit()]
        if len(meaningful) == 0 and len(words) <= 3:
            continue
        cleaned.append(stripped)
    result = '\n'.join(cleaned)
    result = re.sub(r' +', ' ', result)
    result = re.sub(r'\n{3,}', '\n\n', result)
    lines2 = result.split('\n')
    cleaned2 = []
    for line in lines2:
        line = re.sub(r'^[a-zA-Z]\s+', '', line)
        line = re.sub(r'\s+[a-zA-Z]$', '', line)
        if line.strip():
            cleaned2.append(line.strip())
    return '\n'.join(cleaned2)


def validate_content(cleaned_text):
    words = cleaned_text.split()
    meaningful_words = [w for w in words if len(w) > 2]
    ratio = len(meaningful_words) / len(words) if words else 0
    return {
        'total_words': len(words),
        'meaningful_words': len(meaningful_words),
        'ratio': ratio,
        'is_valid': len(words) >= 200 and ratio >= 0.5
    }


pdf_files = sorted(glob.glob(os.path.join(PDF_DIR, '*.pdf')))
print(f"Total PDF ditemukan: {len(pdf_files)}")

results = []
for idx, pdf_path in enumerate(pdf_files, 1):
    case_id = f"case_{idx:03d}"
    raw_text = extract_clean_text_from_pdf(pdf_path)
    if not raw_text or len(raw_text.strip()) == 0:
        print(f"  [{idx}] GAGAL - teks kosong")
        continue
    clean = post_clean_text(raw_text)
    validation = validate_content(clean)
    status = "OK" if validation['is_valid'] else "LOW"
    print(f"  [{idx:2d}/70] {case_id}: {validation['total_words']} kata, rasio={validation['ratio']:.1%} [{status}]")
    output_path = os.path.join(RAW_DIR, f"{case_id}.txt")
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(clean)
    results.append({
        'case_id': case_id,
        'file': os.path.basename(pdf_path),
        'total_words': validation['total_words'],
        'ratio': validation['ratio'],
        'status': status
    })

ok_count = len([r for r in results if r['status'] == 'OK'])
low_count = len([r for r in results if r['status'] == 'LOW'])
print(f"\nBerhasil: {ok_count}/{len(results)}, Peringatan: {low_count}")
print(f"File tersimpan di: {RAW_DIR}")

---
## Tahap 2: Case Representation - Ekstraksi Metadata dan Fitur

In [ ]:
def parse_nomor_perkara(text):
    m = re.search(r'Nomor\s+([\d]+/[A-Za-z/.]+/\d{4})', text)
    return m.group(1) if m else ''

def parse_nama_terdakwa(text):
    lines = text.split('\n')
    nama_lines = []
    capture = False
    for line in lines:
        stripped = line.strip()
        if stripped.startswith('Nama :') or stripped.startswith('Nama:'):
            capture = True
            nama_lines.append(stripped)
        elif capture and stripped.startswith('Tempat Lahir'):
            break
        elif capture and stripped:
            nama_lines.append(stripped)
        elif capture and not stripped:
            break
    return ' '.join(nama_lines).replace('Nama :', '').replace('Nama:', '').strip()

def parse_pasal(text):
    pasal_pattern = re.findall(
        r'Pasal\s+(\d+(?:\s+Ayat\s+\(\d+\))?(?:\s+(?:juncto|dan|atau)\s+Pasal\s+\d+(?:\s+Ayat\s+\(\d+\))?)*)',
        text, re.IGNORECASE
    )
    unique_pasal = []
    for p in pasal_pattern:
        normalized = re.sub(r'\s+', ' ', p.strip())
        if normalized not in unique_pasal:
            unique_pasal.append(normalized)
    return '; '.join(unique_pasal[:10]) if unique_pasal else ''

def parse_pihak(text):
    jaksa = ''
    terdakwa = ''
    m_jaksa = re.search(r'Penuntut Umum pada (Kejaksaan[^;]+)', text)
    if m_jaksa:
        jaksa = m_jaksa.group(1).strip()
    m_terdakwa = re.search(r'Nama\s*:\s*([^;]+)', text)
    if m_terdakwa:
        terdakwa = m_terdakwa.group(1).strip()
    return f"Jaksa: {jaksa} | Terdakwa: {terdakwa}"

def parse_ringkasan_fakta(text):
    start = text.find('didakwa dengan dakwaan sebagai berikut:')
    if start == -1:
        start = text.find('dakwaan sebagai berikut:')
    if start == -1:
        start = text.find('Dakwaan:')
    end = text.find('Membaca Tuntutan Pidana')
    if end == -1:
        end = text.find('Membaca Putusan')
    if end == -1:
        end = text.find('Mahkamah Agung tersebut;')
    if start != -1 and end != -1 and end > start:
        return text[start:end].strip()
    return ''

def parse_ringkasan_putusan(text):
    markers = ['G A D I L I:', 'G A D I L I;', 'M E N G A D I L I:', 'MENGADILI:', 'G A D I L I']
    found = -1
    for marker in markers:
        pos = text.rfind(marker)
        if pos > found:
            found = pos
    if found == -1:
        return ''
    end_markers = ['Demikianlah diputuskan', 'Demikian diputuskan']
    end_pos = len(text)
    for m in end_markers:
        pos = text.find(m, found)
        if pos != -1 and pos < end_pos:
            end_pos = pos
    return text[found:end_pos].strip()

def parse_pidana_dari_teks(text, amar_text):
    pidana_tahun = 0
    denda_rp = 0
    if amar_text:
        m = re.search(r'pidana penjara (?:selama )?(\d+)', amar_text, re.IGNORECASE)
        if m:
            pidana_tahun = int(m.group(1))
    if pidana_tahun == 0:
        pidana_candidates = re.findall(r'pidana penjara (?:selama )?(\d+)', text, re.IGNORECASE)
        for p in pidana_candidates:
            val = int(p)
            if 1 <= val <= 80:
                pidana_tahun = val
                break
    denda_pattern = re.finditer(
        r'denda\s+(?:terhadap|sebesar|masing-masing)?\s*(?:sebesar\s+)?\s*Rp([\d.,]+)',
        text, re.IGNORECASE
    )
    for m in denda_pattern:
        d_str = m.group(1)
        parts = d_str.split(',')
        integer_part = parts[0].replace('.', '')
        try:
            val = int(integer_part)
            if 50000000 <= val <= 50000000000:
                denda_rp = max(denda_rp, val)
        except ValueError:
            pass
    return pidana_tahun, denda_rp

def parse_tanggal(text):
    m = re.search(r'(?:hari\s+\w+,\s+)?tanggal\s+(\d{1,2}\s+\w+\s+\d{4})', text)
    if m:
        return m.group(1)
    m2 = re.search(r'(\d{1,2}\s+\w+\s+\d{4})\s*telah\s*', text)
    if m2:
        return m2.group(1)
    return ''

raw_files = sorted(glob.glob(os.path.join(RAW_DIR, 'case_*.txt')))
print(f"Total file teks ditemukan: {len(raw_files)}")

cases = []
for idx, filepath in enumerate(raw_files, 1):
    case_id = f"case_{idx:03d}"
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    ringkasan_fakta = parse_ringkasan_fakta(text)
    ringkasan_putusan = parse_ringkasan_putusan(text)
    pidana_tahun, denda_rp = parse_pidana_dari_teks(text, ringkasan_putusan)
    cases.append({
        'case_id': case_id,
        'no_perkara': parse_nomor_perkara(text),
        'tanggal': parse_tanggal(text),
        'nama_terdakwa': parse_nama_terdakwa(text),
        'ringkasan_fakta': ringkasan_fakta,
        'ringkasan_putusan': ringkasan_putusan,
        'pasal': parse_pasal(text),
        'pihak': parse_pihak(text),
        'pidana_penjara_tahun': pidana_tahun,
        'denda_rp': denda_rp,
        'word_count': len(text.split()),
        'text_full': text
    })

df = pd.DataFrame(cases)
df.to_csv(os.path.join(PROCESSED_DIR, 'cases.csv'), index=False, encoding='utf-8')
print(f"\nTotal kasus: {len(df)}")
print(f"Rata-rata panjang teks: {df['word_count'].mean():.0f} kata")
print(f"Rata-rata pidana: {df['pidana_penjara_tahun'].mean():.1f} tahun")
print(f"Kasus dengan denda: {df[df['denda_rp'] > 0].shape[0]}/{len(df)}")
df[['case_id', 'no_perkara', 'pasal', 'pidana_penjara_tahun', 'denda_rp']].head(10)

---
## Tahap 3: Case Retrieval - TF-IDF, Cosine Similarity, SVM & Naive Bayes

**Pendekatan yang diuji:**
1. **TF-IDF + Cosine Similarity** (baseline)
2. **TF-IDF + SVM** (classification-based retrieval)
3. **TF-IDF + Naive Bayes** (classification-based retrieval)

**Splitting:** 80% train / 20% test

In [ ]:
INDONESIAN_STOPWORDS = [
    'dan', 'di', 'ke', 'dari', 'yang', 'ini', 'itu', 'dengan', 'untuk',
    'pada', 'adalah', 'akan', 'telah', 'sudah', 'bisa', 'dapat', 'tidak',
    'atau', 'juga', 'saya', 'kami', 'kita', 'mereka', 'dia', 'ia',
    'oleh', 'sebagai', 'dalam', 'bahwa', 'karena', 'jika', 'maka',
    'serta', 'seperti', 'secara', 'tersebut', 'setelah', 'sebelum',
    'tentang', 'antara', 'ada', 'lebih', 'lain', 'sangat', 'semua',
    'hal', 'sehingga', 'namun', 'tetapi', 'sedangkan', 'meskipun',
    'walaupun', 'perlu', 'harus', 'selalu', 'belum', 'pernah',
    'saat', 'ketika', 'hingga', 'sampai', 'sejak', 'selama', 'tanpa',
    'lagi', 'masih', 'hanya', 'saja', 'pun', 'punya', 'merupakan',
    'ialah', 'yakni', 'yaitu', 'baik', 'maupun', 'ataupun',
    'membaca', 'menimbang', 'mengingat', 'memperhatikan',
    'menetapkan', 'menyatakan', 'menjatuhkan', 'memutuskan',
]

def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

print(f"Kasus tersedia: {len(df)}")
print(f"Membagi data 80:20 (train:test)...")

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
print(f"Train: {len(train_df)} kasus, Test: {len(test_df)} kasus")
print(f"\nData test:")
test_df[['case_id', 'no_perkara', 'pidana_penjara_tahun']].head()

In [ ]:
# Bangun TF-IDF dari data train
train_texts = [preprocess_text(str(t)) for t in train_df['text_full']]
train_case_ids = train_df['case_id'].tolist()

vectorizer = TfidfVectorizer(
    max_features=10000,
    min_df=2,
    max_df=0.90,
    stop_words=INDONESIAN_STOPWORDS,
    ngram_range=(1, 3),
    sublinear_tf=True
)
tfidf_train = vectorizer.fit_transform(train_texts)
print(f"TF-IDF train shape: {tfidf_train.shape}")
print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())}")

# Transform data test dengan vectorizer yang sama
test_texts = [preprocess_text(str(t)) for t in test_df['text_full']]
tfidf_test = vectorizer.transform(test_texts)
test_case_ids = test_df['case_id'].tolist()
print(f"TF-IDF test shape: {tfidf_test.shape}")

### Model 1: TF-IDF + Cosine Similarity (Baseline)

In [ ]:
def retrieve_cosine(query_text, vectorizer, tfidf_matrix, case_ids, k=5):
    """Retrieval berbasis Cosine Similarity."""
    query_clean = preprocess_text(query_text)
    query_vec = vectorizer.transform([query_clean])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_k_idx = similarities.argsort()[-k:][::-1]
    return [{'case_id': case_ids[i], 'score': float(similarities[i])} for i in top_k_idx]

# Simpan model untuk keperluan reuse
model_path = os.path.join(EVAL_DIR, 'tfidf_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump({
        'vectorizer': vectorizer,
        'tfidf_train': tfidf_train,
        'train_case_ids': train_case_ids,
        'train_df': train_df
    }, f)
print(f"Model Cosine Similarity tersimpan: {model_path}")

### Model 2: TF-IDF + SVM (Support Vector Machine)

In [ ]:
# Siapkan label untuk klasifikasi: kategori berdasarkan pasal utama
def extract_pasal_kategori(pasal_text):
    """Ekstrak kategori pasal utama dari teks pasal."""
    if not isinstance(pasal_text, str) or pasal_text == '':
        return 'lainnya'
    pasal_numbers = re.findall(r'Pasal\s+(\d+)', pasal_text)
    if not pasal_numbers:
        return 'lainnya'
    first_pasal = int(pasal_numbers[0])
    if first_pasal <= 4:
        return 'pasal_1-4'
    elif first_pasal <= 12:
        return 'pasal_5-12'
    elif first_pasal <= 17:
        return 'pasal_13-17'
    else:
        return 'pasal_18+'

# Beri label pada data train
train_labels = train_df['pasal'].apply(extract_pasal_kategori)
test_labels = test_df['pasal'].apply(extract_pasal_kategori)

print("Distribusi label train:")
print(train_labels.value_counts())

# Latih SVM
svm_model = LinearSVC(random_state=42, max_iter=5000, C=1.0)
svm_model.fit(tfidf_train, train_labels)
print(f"\nSVM dilatih dengan {tfidf_train.shape[0]} data train")

# Prediksi pada data test
svm_pred = svm_model.predict(tfidf_test)
print(f"Akurasi SVM pada test: {accuracy_score(test_labels, svm_pred):.3f}")
print(f"F1-Score SVM pada test: {f1_score(test_labels, svm_pred, average='weighted'):.3f}")

### Model 3: TF-IDF + Naive Bayes

In [ ]:
# Latih Naive Bayes
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(tfidf_train, train_labels)
print(f"Naive Bayes dilatih dengan {tfidf_train.shape[0]} data train")

# Prediksi pada data test
nb_pred = nb_model.predict(tfidf_test)
print(f"Akurasi Naive Bayes pada test: {accuracy_score(test_labels, nb_pred):.3f}")
print(f"F1-Score Naive Bayes pada test: {f1_score(test_labels, nb_pred, average='weighted'):.3f}")

### Fungsi Retrieval untuk Ketiga Model

In [ ]:
def retrieve_svm(query_text, vectorizer, tfidf_all, case_ids, svm_model, k=5):
    """Retrieval menggunakan SVM: prediksi kategori, lalu cari top-k dalam kategori itu."""
    query_clean = preprocess_text(query_text)
    query_vec = vectorizer.transform([query_clean])
    predicted_label = svm_model.predict(query_vec)[0]
    label_scores = svm_model.decision_function(query_vec)[0]
    similarities = cosine_similarity(query_vec, tfidf_all).flatten()
    top_k_idx = similarities.argsort()[-k*3:][::-1]
    results = []
    for i in top_k_idx:
        results.append({'case_id': case_ids[i], 'score': float(similarities[i])})
    return results[:k], predicted_label

def retrieve_nb(query_text, vectorizer, tfidf_all, case_ids, nb_model, k=5):
    """Retrieval menggunakan Naive Bayes: prediksi kategori, lalu cari top-k dalam kategori itu."""
    query_clean = preprocess_text(query_text)
    query_vec = vectorizer.transform([query_clean])
    predicted_label = nb_model.predict(query_vec)[0]
    similarities = cosine_similarity(query_vec, tfidf_all).flatten()
    top_k_idx = similarities.argsort()[-k*3:][::-1]
    results = []
    for i in top_k_idx:
        results.append({'case_id': case_ids[i], 'score': float(similarities[i])})
    return results[:k], predicted_label

print("Fungsi retrieval untuk ketiga model siap.")

### Buat Query Uji dan Jalankan Retrieval

In [ ]:
# Buat query dari data test (leave-one-out pada data test)
queries = []
for i in range(min(5, len(test_df))):
    row = test_df.iloc[i]
    fakta = str(row['ringkasan_fakta']) if pd.notna(row['ringkasan_fakta']) else ''
    pasal = str(row['pasal']) if pd.notna(row['pasal']) else ''
    queries.append({
        'query_id': f'query_{i+1:02d}',
        'query_text': fakta[:2000] if fakta else str(row['text_full'])[:2000],
        'source_case': row['case_id'],
        'ground_truth': [row['case_id']],
        'type': 'leave_one_out'
    })

# Query sintetis
synthetic_queries = [
    {'query_id': 'query_06', 'query_text': 'Turut serta melakukan melaksanakan penempatan pekerja migran Indonesia. Pekerja migran Indonesia ditempatkan di luar negeri tanpa hak dan dieksploitasi. Pasal 4 juncto Pasal 10 Undang-Undang Nomor 21 Tahun 2007.', 'ground_truth': ['case_004'], 'type': 'synthetic'},
    {'query_id': 'query_07', 'query_text': 'Terdakwa bekerja sebagai manajer di klub malam yang mempekerjakan anak di bawah umur sebagai waitress. Anak korban dieksploitasi secara ekonomi. Pasal 88 Ayat (1) juncto Pasal 76 I Undang-Undang Perlindungan Anak.', 'ground_truth': ['case_001'], 'type': 'synthetic'},
    {'query_id': 'query_08', 'query_text': 'Terdakwa menawarkan jasa prostitusi melalui aplikasi MiChat. Terdakwa mencari tamu untuk melakukan persetubuhan dengan para saksi dan mendapatkan fee. Pasal 12 Undang-Undang Nomor 21 Tahun 2007.', 'ground_truth': ['case_002'], 'type': 'synthetic'},
    {'query_id': 'query_09', 'query_text': 'Mahasiswa magang dikirim ke Jepang untuk program magang tetapi dipaksa bekerja melebihi jam kerja. Visa diubah dari visa pelajar menjadi visa pekerja. Pasal 4 juncto Pasal 48 Undang-Undang Nomor 21 Tahun 2007.', 'ground_truth': ['case_070'], 'type': 'synthetic'},
    {'query_id': 'query_10', 'query_text': 'Terdakwa merekrut anak di bawah umur untuk dieksploitasi secara seksual. Anak korban berusia 16 tahun dan 13 tahun. Terdakwa mencarikan tamu laki-laki untuk anak korban. Pasal 2 Ayat (1) juncto Pasal 17 Undang-Undang Nomor 21 Tahun 2007.', 'ground_truth': ['case_017'], 'type': 'synthetic'}
]
queries.extend(synthetic_queries)

print(f"Total query uji: {len(queries)} (leave_one_out: 5, synthetic: 5)")

# Simpan queries
with open(os.path.join(EVAL_DIR, 'queries.json'), 'w', encoding='utf-8') as f:
    json.dump(queries, f, indent=2, ensure_ascii=False)

In [ ]:
# Gunakan SEMUA data sebagai corpus untuk retrieval (termasuk train)
all_texts = [preprocess_text(str(t)) for t in df['text_full']]
all_case_ids = df['case_id'].tolist()
tfidf_all = vectorizer.transform(all_texts)

retrieval_results = []
for q in queries:
    # Cosine Similarity
    res_cos = retrieve_cosine(q['query_text'], vectorizer, tfidf_all, all_case_ids, k=5)
    # SVM
    res_svm, svm_label = retrieve_svm(q['query_text'], vectorizer, tfidf_all, all_case_ids, svm_model, k=5)
    # Naive Bayes
    res_nb, nb_label = retrieve_nb(q['query_text'], vectorizer, tfidf_all, all_case_ids, nb_model, k=5)

    gt = q['ground_truth']
    result = {
        'query_id': q['query_id'],
        'type': q['type'],
        'ground_truth': gt,
        'cosine_top5': [r['case_id'] for r in res_cos],
        'cosine_scores': [r['score'] for r in res_cos],
        'svm_top5': [r['case_id'] for r in res_svm],
        'svm_label': svm_label,
        'nb_top5': [r['case_id'] for r in res_nb],
        'nb_label': nb_label,
    }

    # Hitung metrik per model
    for model_name, key in [('cosine', 'cosine_top5'), ('svm', 'svm_top5'), ('nb', 'nb_top5')]:
        retrieved = set(result[key])
        hits = len(retrieved & set(gt))
        precision = hits / len(retrieved) if retrieved else 0
        recall = hits / len(gt) if gt else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        mrr = 0
        for rank, rid in enumerate(result[key], 1):
            if rid in gt:
                mrr = 1.0 / rank
                break
        result[f'{model_name}_precision'] = precision
        result[f'{model_name}_recall'] = recall
        result[f'{model_name}_f1'] = f1
        result[f'{model_name}_mrr'] = mrr

    retrieval_results.append(result)

    print(f"{q['query_id']} ({q['type']})")
    print(f"  Cosine: {result['cosine_top5'][:3]}... P={result['cosine_precision']:.2f} R={result['cosine_recall']:.2f} MRR={result['cosine_mrr']:.2f}")
    print(f"  SVM:    {result['svm_top5'][:3]}... P={result['svm_precision']:.2f} R={result['svm_recall']:.2f} MRR={result['svm_mrr']:.2f}")
    print(f"  NB:     {result['nb_top5'][:3]}... P={result['nb_precision']:.2f} R={result['nb_recall']:.2f} MRR={result['nb_mrr']:.2f}")

# Simpan hasil retrieval
with open(os.path.join(EVAL_DIR, 'retrieval_results.json'), 'w', encoding='utf-8') as f:
    json.dump(retrieval_results, f, indent=2, ensure_ascii=False)

print("\nHasil retrieval tersimpan.")

---
## Tahap 4: Case Solution Reuse - Prediksi Hasil Putusan

In [ ]:
def predict_outcome_weighted(query_text, vectorizer, tfidf_matrix, case_ids, case_df, k=5):
    """Prediksi menggunakan Weighted Similarity Voting."""
    results = retrieve_cosine(query_text, vectorizer, tfidf_matrix, case_ids, k)
    total_weight = sum(r['score'] for r in results)
    if total_weight == 0:
        return {'predicted_pidana': 0, 'predicted_denda': 0, 'top_k_cases': [], 'top_k_scores': []}
    weighted_pidana = 0.0
    weighted_denda = 0.0
    for r in results:
        case_data = case_df[case_df['case_id'] == r['case_id']]
        if case_data.empty:
            continue
        row = case_data.iloc[0]
        weight = r['score'] / total_weight
        weighted_pidana += weight * float(row['pidana_penjara_tahun'])
        weighted_denda += weight * float(row['denda_rp'])
    return {
        'predicted_pidana': round(weighted_pidana, 1),
        'predicted_denda': round(weighted_denda, -3),
        'top_k_cases': [r['case_id'] for r in results],
        'top_k_scores': [r['score'] for r in results]
    }

predictions = []
for q in queries:
    result = predict_outcome_weighted(q['query_text'], vectorizer, tfidf_all, all_case_ids, df, k=5)
    ground_pidana = 0
    ground_denda = 0
    if q['type'] == 'leave_one_out':
        src = df[df['case_id'] == q['source_case']]
        if not src.empty:
            ground_pidana = int(src.iloc[0]['pidana_penjara_tahun'])
            ground_denda = int(src.iloc[0]['denda_rp'])
    else:
        gt = q['ground_truth']
        if gt:
            gt_data = df[df['case_id'].isin(gt)]
            if not gt_data.empty:
                ground_pidana = int(gt_data.iloc[0]['pidana_penjara_tahun'])
                ground_denda = int(gt_data.iloc[0]['denda_rp'])
    predictions.append({
        'query_id': q['query_id'],
        'type': q['type'],
        'predicted_pidana': result['predicted_pidana'],
        'predicted_denda': result['predicted_denda'],
        'ground_truth_pidana': ground_pidana,
        'ground_truth_denda': ground_denda,
        'top_5_case_ids': '; '.join(result['top_k_cases']),
        'method': 'weighted_voting'
    })
    print(f"{q['query_id']}: Pred={result['predicted_pidana']} thn, Ground={ground_pidana} thn")

pred_df = pd.DataFrame(predictions)
pred_df.to_csv(os.path.join(RESULTS_DIR, 'predictions.csv'), index=False, encoding='utf-8')
print(f"\nPrediksi tersimpan.")

---
## Tahap 5: Evaluasi Model dan Visualisasi

In [ ]:
# Ringkasan metrik per model
models = ['cosine', 'svm', 'nb']
model_names = {'cosine': 'TF-IDF + Cosine', 'svm': 'TF-IDF + SVM', 'nb': 'TF-IDF + Naive Bayes'}

summary = []
for m in models:
    prec = np.mean([r[f'{m}_precision'] for r in retrieval_results])
    rec = np.mean([r[f'{m}_recall'] for r in retrieval_results])
    f1 = np.mean([r[f'{m}_f1'] for r in retrieval_results])
    mrr = np.mean([r[f'{m}_mrr'] for r in retrieval_results])
    summary.append({
        'Model': model_names[m],
        'Precision@5': round(prec, 3),
        'Recall@5': round(rec, 3),
        'F1-Score': round(f1, 3),
        'MRR': round(mrr, 3)
    })

summary_df = pd.DataFrame(summary)
print("=" * 70)
print("RINGKASAN METRIK RETRIEVAL PER MODEL")
print("=" * 70)
print(summary_df.to_string(index=False))

summary_df.to_csv(os.path.join(EVAL_DIR, 'retrieval_metrics.csv'), index=False)

In [ ]:
# Evaluasi prediksi
valid = pred_df[pred_df['ground_truth_pidana'] > 0]
if not valid.empty:
    mae_pidana = mean_absolute_error(valid['ground_truth_pidana'], valid['predicted_pidana'])
    mae_denda = mean_absolute_error(valid['ground_truth_denda'], valid['predicted_denda'])
    print(f"MAE Pidana: {mae_pidana:.2f} tahun")
    print(f"MAE Denda: Rp{mae_denda:,.0f}")

    valid_copy = valid.copy()
    valid_copy['error_pidana'] = abs(valid_copy['predicted_pidana'] - valid_copy['ground_truth_pidana'])
    valid_copy['error_denda'] = abs(valid_copy['predicted_denda'] - valid_copy['ground_truth_denda'])
    valid_copy.to_csv(os.path.join(EVAL_DIR, 'prediction_metrics.csv'), index=False)

    print(f"\nError Analysis:")
    print(f"  Error pidana <= 2 tahun: {(valid_copy['error_pidana'] <= 2).sum()}/{len(valid_copy)}")
    print(f"  Error pidana <= 5 tahun: {(valid_copy['error_pidana'] <= 5).sum()}/{len(valid_copy)}")
    
    print(f"\nTop 3 kesalahan terbesar:")
    worst = valid_copy.nlargest(3, 'error_pidana')
    for _, row in worst.iterrows():
        print(f"  {row['query_id']}: Pred={row['predicted_pidana']:.1f} thn, Ground={row['ground_truth_pidana']:.0f} thn, Error={row['error_pidana']:.1f}")

In [ ]:
# Visualisasi perbandingan metrik
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
metrics_to_plot = ['Precision@5', 'Recall@5', 'F1-Score', 'MRR']
colors = ['#2196F3', '#FF9800', '#4CAF50']

for i, metric in enumerate(metrics_to_plot):
    ax = axes[i]
    bars = ax.bar(summary_df['Model'], summary_df[metric], color=colors, width=0.5)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, summary_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Perbandingan Metrik Retrieval per Model', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'metrik_perbandingan.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot tersimpan: {os.path.join(EVAL_DIR, 'metrik_perbandingan.png')}")

In [ ]:
# Akurasi klasifikasi SVM vs Naive Bayes
print("=" * 50)
print("AKURASI KLASIFIKASI (TF-IDF)")
print("=" * 50)
print(f"SVM:          {accuracy_score(test_labels, svm_pred):.3f} (F1: {f1_score(test_labels, svm_pred, average='weighted'):.3f})")
print(f"Naive Bayes:  {accuracy_score(test_labels, nb_pred):.3f} (F1: {f1_score(test_labels, nb_pred, average='weighted'):.3f})")

# Bar chart perbandingan akurasi
fig, ax = plt.subplots(figsize=(6, 4))
models_cls = ['SVM', 'Naive Bayes']
accuracies = [accuracy_score(test_labels, svm_pred), accuracy_score(test_labels, nb_pred)]
f1_scores = [f1_score(test_labels, svm_pred, average='weighted'), f1_score(test_labels, nb_pred, average='weighted')]

x = np.arange(len(models_cls))
width = 0.35
bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#2196F3')
bars2 = ax.bar(x + width/2, f1_scores, width, label='F1-Score', color='#FF9800')

ax.set_ylabel('Skor')
ax.set_title('Akurasi Klasifikasi TF-IDF', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models_cls)
ax.legend()
ax.set_ylim(0, 1.1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02, f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02, f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, 'klasifikasi_akurasi.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Ringkasan Hasil Akhir

In [ ]:
print("=" * 70)
print("RINGKASAN HASIL EVALUASI SISTEM CBR")
print("=" * 70)
print(f"\nDataset: {len(df)} putusan pengadilan TPPO")
print(f"Train/Test Split: {len(train_df)}/{len(test_df)} (80/20)")
print(f"Query uji: {len(queries)} (leave_one_out: 5, synthetic: 5)")

print(f"\n--- Retrieval (3 Model) ---")
print(summary_df.to_string(index=False))

print(f"\n--- Klasifikasi ---")
print(f"SVM Accuracy: {accuracy_score(test_labels, svm_pred):.3f}, F1: {f1_score(test_labels, svm_pred, average='weighted'):.3f}")
print(f"NB Accuracy:  {accuracy_score(test_labels, nb_pred):.3f}, F1: {f1_score(test_labels, nb_pred, average='weighted'):.3f}")

if not valid.empty:
    print(f"\n--- Prediksi (Weighted Voting) ---")
    print(f"MAE Pidana: {mae_pidana:.2f} tahun")
    print(f"MAE Denda: Rp{mae_denda:,.0f}")

print(f"\nFile output:")
print(f"  data/raw/*.txt                  : {len(df)} teks bersih")
print(f"  data/processed/cases.csv        : metadata terstruktur")
print(f"  data/eval/queries.json          : {len(queries)} query uji")
print(f"  data/eval/retrieval_metrics.csv : metrik per model")
print(f"  data/eval/prediction_metrics.csv: metrik prediksi")
print(f"  data/results/predictions.csv     : hasil prediksi")
print(f"  data/eval/metrik_perbandingan.png: plot perbandingan")